# GNN Training Notebook

Trains all three GNN configurations and saves checkpoints to `data/models/gnn/checkpoints/`.

| Model | Country features | Capability edges | Checkpoint |
|-------|-----------------|-----------------|------------|
| **GNN-4F** | 4 BACI features | None | `gnn_4f.pt` |
| **GNN-11F** | 11 BACI+WDI features | None | `gnn_11f.pt` |
| **GNN-11F+LLM** | 11 BACI+WDI features | Top-20 FinLang product similarity | `gnn_11f_llm.pt` |

Run this notebook first, then `evaluation.ipynb` to load checkpoints and produce the comparison table.

In [8]:
import os, sys, pickle, warnings, gc

# Set CUDA memory allocation strategy before importing torch
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import precision_recall_curve, auc
from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv, to_hetero
from torch.utils.checkpoint import checkpoint
warnings.filterwarnings('ignore')
torch.manual_seed(42); np.random.seed(42)

# CUDA memory: disable caching allocator overhead, use conservative allocation
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.set_per_process_memory_fraction(0.95)  # Use ~95% of available VRAM

DATA_DIR  = 'data'
CKPT_DIR  = os.path.join(DATA_DIR, 'models', 'gnn', 'checkpoints')
TRAIN_CUTOFF = 2012
VAL_YEAR  = 2013
TEST_YEAR = 2015
DEVICE    = 'cuda'  # Force GPU to avoid OOM
HIDDEN    = 128
EPOCHS    = 80
LR        = 1e-3
WD        = 1e-5
PATIENCE  = 15
os.makedirs(CKPT_DIR, exist_ok=True)
print(f'Device: {DEVICE}  |  Checkpoints -> {CKPT_DIR}')

Device: cuda  |  Checkpoints -> data\models\gnn\checkpoints


## Load pipeline artifacts

In [9]:
edge_idx_raw   = torch.load(os.path.join(DATA_DIR, 'edge_index_by_year.pt'), weights_only=False)
edge_idx_by_yr = {k: v.long() for k, v in edge_idx_raw.items()}
p_x_by_yr      = torch.load(os.path.join(DATA_DIR, 'product_x_by_year.pt'),  weights_only=False)
c_x_11feat     = torch.load(os.path.join(DATA_DIR, 'country_x_by_year.pt'),  weights_only=False)

with open(os.path.join(DATA_DIR, 'country_mapping.pkl'), 'rb') as f: c_map = pickle.load(f)
with open(os.path.join(DATA_DIR, 'product_mapping.pkl'), 'rb') as f: p_map = pickle.load(f)

train_lbl = pd.read_csv(os.path.join(DATA_DIR, 'train_labels.csv'))
val_lbl   = pd.read_csv(os.path.join(DATA_DIR, 'val_labels.csv'))
test_lbl  = pd.read_csv(os.path.join(DATA_DIR, 'test_labels.csv'))

c_feat_df = pd.read_csv(os.path.join(DATA_DIR, 'country_features.csv'))
BACI_COLS = ['log_export', 'n_products', 'avg_rca', 'max_rca']
c_x_4feat = {}
for yr in sorted(c_feat_df['year'].unique()):
    yd = c_feat_df[c_feat_df['year'] == yr].copy()
    yd['idx'] = yd['country'].map(c_map['to_idx'])
    yd = yd.dropna(subset=['idx']).sort_values('idx')
    c_x_4feat[int(yr)] = torch.tensor(yd[BACI_COLS].values, dtype=torch.float32)

print(f'4-feat:  {c_x_4feat[TEST_YEAR].shape}')
print(f'11-feat: {c_x_11feat[TEST_YEAR].shape}')
print(f'Product: {p_x_by_yr[TEST_YEAR].shape}')
print(f'Train {len(train_lbl):,}  Val {len(val_lbl):,}  Test {len(test_lbl):,}')

4-feat:  torch.Size([233, 4])
11-feat: torch.Size([233, 11])
Product: torch.Size([5018, 3])
Train 1,699,206  Val 128,278  Test 127,531


## Build Capability Edge Index (top-K=20, FinLang)

Connects each product to its 20 most semantically similar products using `FinLang/finance-embeddings-investopedia` embeddings (768-dim).

FinLang discrimination: horses vs diodes = 0.29 (correctly far apart); petrol cars 1500cc vs 3000cc = 0.998 (correctly close). Only ~1% of all pairs exceed cosine 0.70.

In [10]:
EMB_PATH = os.path.join(DATA_DIR, 'product_llm_embeddings.pt')
CAP_PATH = os.path.join(DATA_DIR, 'capability_edge_index.pt')

if os.path.exists(CAP_PATH):
    cap_ei = torch.load(CAP_PATH, weights_only=False).long()
    print(f'Loaded: {cap_ei.shape}')
else:
    print('Building top-20 capability edges...')
    emb  = torch.load(EMB_PATH, weights_only=False)
    E_np = emb.numpy()
    P, K = E_np.shape[0], 20
    sim  = E_np @ E_np.T
    np.fill_diagonal(sim, -1.0)
    topk = np.argsort(-sim, axis=1)[:, :K]
    src = topk.flatten()
    dst = np.repeat(np.arange(P), K)
    all_src = np.concatenate([src, dst])
    all_dst = np.concatenate([dst, src])
    pairs   = np.stack([all_src, all_dst], axis=0)
    pairs   = np.unique(pairs, axis=1)
    cap_ei  = torch.tensor(pairs, dtype=torch.long)
    torch.save(cap_ei, CAP_PATH)
    print(f'Saved -> {CAP_PATH}')

print(f'Capability edges: {tuple(cap_ei.shape)}  ({cap_ei.shape[1]:,} edges)')

Loaded: torch.Size([2, 144192])
Capability edges: (2, 144192)  (144,192 edges)


## Model Architecture & Training Helpers

- **BipartiteEncoder**: Linear projections then 2-layer SAGEConv via `to_hetero`
- **TemporalGNN**: encoder on each of 5 annual snapshots then GRU per node type
- **LinkPredictor**: MLP on [z_country || z_product]

`build_snap` = no capability edges. `build_snap_llm` = adds `('product','capability','product')` edges.

In [ ]:
class _HomoGNN(nn.Module):
    def __init__(self, hidden, drop=0.3):
        super().__init__()
        self.c1 = SAGEConv(hidden, hidden)
        self.c2 = SAGEConv(hidden, hidden)
        self.drop = drop
    def forward(self, x, edge_index):
        x = F.dropout(self.c1(x, edge_index).relu(), p=self.drop, training=self.training)
        return self.c2(x, edge_index)

class BipartiteEncoder(nn.Module):
    def __init__(self, c_in, hidden, meta):
        super().__init__()
        self.country_lin = nn.Linear(c_in, hidden)
        self.product_lin = nn.Linear(3, hidden)
        self.gnn = to_hetero(_HomoGNN(hidden), meta)
    def forward(self, x_dict, ei_dict):
        return self.gnn({'country': self.country_lin(x_dict['country']),
                         'product': self.product_lin(x_dict['product'])}, ei_dict)

class TemporalGNN(nn.Module):
    def __init__(self, enc, hidden):
        super().__init__()
        self.enc   = enc
        self.gru_c = nn.GRU(hidden, hidden, batch_first=False)
        self.gru_p = nn.GRU(hidden, hidden, batch_first=False)
    def forward(self, snaps, use_checkpoint=False):
        cs, ps = [], []
        for s in snaps:
            if use_checkpoint and self.training:
                z = checkpoint(self.enc, s.x_dict, s.edge_index_dict, use_reentrant=False)
            else:
                z = self.enc(s.x_dict, s.edge_index_dict)
            cs.append(z['country']); ps.append(z['product'])
        z_c, _ = self.gru_c(torch.stack(cs))
        z_p, _ = self.gru_p(torch.stack(ps))
        return {'country': z_c[-1], 'product': z_p[-1]}

class LinkPredictor(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(hidden * 2, hidden), nn.ReLU(), nn.Dropout(0.2), nn.Linear(hidden, 1))
    def forward(self, zc, zp, ei):
        return self.mlp(torch.cat([zc[ei[0]], zp[ei[1]]], -1)).view(-1)

def build_snap(year, c_x):
    d = HeteroData()
    d['country'].x = c_x[year]
    d['product'].x = p_x_by_yr[year]
    ei = edge_idx_by_yr[year].long()
    d['country', 'exports',     'product'].edge_index = ei
    d['product', 'rev_exports', 'country'].edge_index = ei.flip(0)
    return d

def build_snap_llm(year, c_x, cap_edge_index):
    d = build_snap(year, c_x)
    d['product', 'capability', 'product'].edge_index = cap_edge_index.long()
    return d

def build_sample(obs_yr, ldf, c_x, use_llm=False, cap_edge_index=None):
    fn = (lambda y: build_snap_llm(y, c_x, cap_edge_index)) if use_llm else (lambda y: build_snap(y, c_x))
    snaps = [fn(y) for y in range(obs_yr - 4, obs_yr + 1)]
    row   = ldf[ldf['year'] == obs_yr].copy().reset_index(drop=True)
    ci_s  = row['country'].map(c_map['to_idx'])
    pi_s  = row['product'].map(p_map['to_idx'])
    ok    = ci_s.notna() & pi_s.notna()
    ci, pi = ci_s[ok].astype(int).values, pi_s[ok].astype(int).values
    return {
        'snapshots': snaps,
        'labels': {'edge_label_index': torch.tensor([ci, pi], dtype=torch.long),
                   'edge_label':       torch.tensor(row.loc[ok, 'label'].values, dtype=torch.float32)},
        'year':          int(obs_yr),
        'countries_raw': row.loc[ok, 'country'].values,
        'products_raw':  row.loc[ok, 'product'].values,
    }

def to_dev(samp, dev):
    for s in samp['snapshots']:
        s['country'].x = s['country'].x.to(dev)
        s['product'].x = s['product'].x.to(dev)
        for et in s.edge_types:
            s[et].edge_index = s[et].edge_index.to(device=dev, dtype=torch.long)
    samp['labels']['edge_label_index'] = samp['labels']['edge_label_index'].to(dev)
    samp['labels']['edge_label']       = samp['labels']['edge_label'].to(dev)

def from_dev(samp):
    """Move sample tensors back to CPU after each training step to free GPU memory."""
    for s in samp['snapshots']:
        s['country'].x = s['country'].x.cpu()
        s['product'].x = s['product'].x.cpu()
        for et in s.edge_types:
            s[et].edge_index = s[et].edge_index.cpu()
    samp['labels']['edge_label_index'] = samp['labels']['edge_label_index'].cpu()
    samp['labels']['edge_label']       = samp['labels']['edge_label'].cpu()

@torch.no_grad()
def get_scores(mdl, pred, samp, dev):
    mdl.eval(); pred.eval()
    to_dev(samp, dev)
    z      = mdl(samp['snapshots'], use_checkpoint=False)
    scores = torch.sigmoid(pred(z['country'], z['product'],
                                samp['labels']['edge_label_index'])).cpu().numpy()
    from_dev(samp)
    return scores

def train_gnn(name, tr, va, c_in, hidden=HIDDEN, epochs=EPOCHS,
              patience=PATIENCE, dev=DEVICE, save_path=None):
    meta = tr[0]['snapshots'][0].metadata()
    enc  = BipartiteEncoder(c_in, hidden, meta).to(dev)
    mdl  = TemporalGNN(enc, hidden).to(dev)
    pred = LinkPredictor(hidden).to(dev)
    all_lv = torch.cat([s['labels']['edge_label'] for s in tr])
    n_pos  = all_lv.sum().item()
    n_neg  = (all_lv == 0).sum().item()
    pw     = torch.tensor([n_neg / max(n_pos, 1)], device=dev)
    crit   = nn.BCEWithLogitsLoss(pos_weight=pw)
    opt    = torch.optim.Adam(list(mdl.parameters()) + list(pred.parameters()),
                               lr=LR, weight_decay=WD)
    best_vpa, best_state, no_imp = -1.0, None, 0
    print(f'Training {name}  |  c_in={c_in}  |  pos_weight={n_neg/n_pos:.1f}x  |  Grad-ckpt: ON  |  FP32')
    for ep in range(1, epochs + 1):
        mdl.train(); pred.train(); ep_loss = 0.0
        for samp in tr:
            to_dev(samp, dev)
            opt.zero_grad()
            z    = mdl(samp['snapshots'], use_checkpoint=True)
            loss = crit(pred(z['country'], z['product'], samp['labels']['edge_label_index']),
                        samp['labels']['edge_label'])
            loss.backward()
            nn.utils.clip_grad_norm_(list(mdl.parameters()) + list(pred.parameters()), 1.0)
            opt.step()
            ep_loss += loss.item()
            from_dev(samp)
            del z, loss

        vscores = get_scores(mdl, pred, va, dev)
        vlabels = va['labels']['edge_label'].numpy()
        vp, vr, _ = precision_recall_curve(vlabels, vscores)
        vpa = auc(vr, vp)
        if vpa > best_vpa:
            best_vpa = vpa
            best_state = ({k: v.cpu().clone() for k, v in mdl.state_dict().items()},
                          {k: v.cpu().clone() for k, v in pred.state_dict().items()})
            no_imp = 0
        else:
            no_imp += 1
        if ep % 10 == 0:
            print(f'  Ep {ep:3d}  loss={ep_loss/len(tr):.4f}  val_PR-AUC={vpa:.4f}  best={best_vpa:.4f}')
        if no_imp >= patience:
            print(f'  Early stop ep {ep}  (best val PR-AUC={best_vpa:.4f})')
            break

    mdl.load_state_dict({k: v.to(dev) for k, v in best_state[0].items()})
    pred.load_state_dict({k: v.to(dev) for k, v in best_state[1].items()})
    if save_path:
        torch.save({'mdl_state':  {k: v.cpu() for k, v in mdl.state_dict().items()},
                    'pred_state': {k: v.cpu() for k, v in pred.state_dict().items()},
                    'c_in': c_in, 'hidden': hidden, 'meta': meta}, save_path)
        print(f'  Checkpoint -> {save_path}')
    return mdl, pred

print('Architecture and helpers loaded.')

## Train GNN-4F (BACI only)

In [12]:
CKPT_4F = os.path.join(CKPT_DIR, 'gnn_4f.pt')
torch.manual_seed(42); np.random.seed(42)

if os.path.exists(CKPT_4F):
    print(f'Already exists: {CKPT_4F} — delete to re-train.')
else:
    tr4 = [build_sample(y, train_lbl, c_x_4feat) for y in sorted(train_lbl['year'].unique())]
    va4 = build_sample(VAL_YEAR, val_lbl, c_x_4feat)
    train_gnn('GNN-4F', tr4, va4, c_in=4, save_path=CKPT_4F)
print('GNN-4F done.')

Already exists: data\models\gnn\checkpoints\gnn_4f.pt — delete to re-train.
GNN-4F done.


## Train GNN-11F (BACI + WDI)

In [13]:
CKPT_11F = os.path.join(CKPT_DIR, 'gnn_11f.pt')
torch.manual_seed(42); np.random.seed(42)

if os.path.exists(CKPT_11F):
    print(f'Already exists: {CKPT_11F} — delete to re-train.')
else:
    tr11 = [build_sample(y, train_lbl, c_x_11feat) for y in sorted(train_lbl['year'].unique())]
    va11 = build_sample(VAL_YEAR, val_lbl, c_x_11feat)
    train_gnn('GNN-11F', tr11, va11, c_in=11, save_path=CKPT_11F)
print('GNN-11F done.')

Already exists: data\models\gnn\checkpoints\gnn_11f.pt — delete to re-train.
GNN-11F done.


## Train GNN-11F+LLM (BACI + WDI + FinLang capability edges)

Adds `('product','capability','product')` edges to every snapshot derived from top-20 FinLang nearest-neighbour product similarity. This is the core H2 innovation — the GNN can now propagate information between semantically similar products, not just through shared-exporting countries.

In [14]:
CKPT_LLM = os.path.join(CKPT_DIR, 'gnn_11f_llm.pt')
torch.manual_seed(42); np.random.seed(42)

if os.path.exists(CKPT_LLM):
    print(f'Already exists: {CKPT_LLM} — delete to re-train.')
else:
    tr_llm = [build_sample(y, train_lbl, c_x_11feat, use_llm=True, cap_edge_index=cap_ei)
              for y in sorted(train_lbl['year'].unique())]
    va_llm = build_sample(VAL_YEAR, val_lbl, c_x_11feat, use_llm=True, cap_edge_index=cap_ei)
    train_gnn('GNN-11F+LLM', tr_llm, va_llm, c_in=11, save_path=CKPT_LLM)
print('GNN-11F+LLM done.')

Training GNN-11F+LLM  |  c_in=11  |  pos_weight=5.0x  |  Gradient Checkpointing: ON
  Ep  10  loss=1.1345  val_PR-AUC=0.2393  best=0.2709
  Early stop ep 16  (best val PR-AUC=0.2709)
  Checkpoint -> data\models\gnn\checkpoints\gnn_11f_llm.pt
GNN-11F+LLM done.


## Summary

In [15]:
print('Checkpoint summary:')
for name, path in [('GNN-4F', CKPT_4F), ('GNN-11F', CKPT_11F), ('GNN-11F+LLM', CKPT_LLM)]:
    exists = os.path.exists(path)
    size   = f'{os.path.getsize(path)/1e6:.1f} MB' if exists else 'missing'
    status = 'READY' if exists else 'NOT FOUND'
    print(f'  {name:<18} {status}  ({size})')
print()
print('Run evaluation.ipynb to load checkpoints and produce the comparison table.')

Checkpoint summary:
  GNN-4F             READY  (1.5 MB)
  GNN-11F            READY  (1.5 MB)
  GNN-11F+LLM        READY  (1.7 MB)

Run evaluation.ipynb to load checkpoints and produce the comparison table.
